# Gasificador 0D — Optimización y control avanzado

> **Estado:** pendiente de implementación.
> Prerequisito: `test_gasifier_03_0D_signals.ipynb` completado y validado.

Este notebook cubrirá técnicas de optimización y control avanzado sobre el gasificador 0D,
usando el framework de señales ya implementado en `dev/gasifier`.

---

## Contenido planificado

### Bloque 4A — Controlador PID en el runner

Implementar un controlador PID con acumulador integral en el runner (no en el RHS)
para evitar la acumulación espuria durante la estimación del Jacobiano BDF.

```python
# Patrón PID en el runner — ver .claude/equipment/signal-bc-integration.md
_integral = [0.0]
_t_prev   = [0.0]

def pid_T_wall(t, snap):
    error  = SP - snap.get("Ts_mean", SP)
    dt     = t - _t_prev[0]
    _integral[0] += error * dt
    _t_prev[0]    = t
    return bias + Kp*error + Ki*_integral[0]
```

### Bloque 4B — Barrido paramétrico (`parametric_sweep`)

Barrer valores de T_wall × mc_wb × t_end para mapear el espacio de conversión.
Usar `src.control.optimization.parametric_sweep` cuando esté implementado.

```python
from src.control.optimization import parametric_sweep

results = parametric_sweep(
    base_params=params_base,
    sweep_vars={
        "T_wall":  [700.0, 900.0, 1073.15],  # [K]
        "mc_wb":   [0.10, 0.165, 0.25],       # [-]
    },
    objective_fn=lambda col: {
        "conv_bio":   1.0 - col._rho_s_results[-1, 0, 0] / col._rho_s_results[0, 0, 0],
        "Ts_final":   col._Ts_results[-1, 0],
        "P_max":      col._P_results[:, 0].max(),
    },
    solver_config=solver_cfg,
)
```

### Bloque 4C — Optimización de T_wall para maximizar conversión de biomasa

Encontrar el perfil de T_wall(t) que maximiza la conversión total a t=3600 s
sujeto a P_max < 1.5 bar.

```python
from src.control.optimization import optimize_bc

result_opt = optimize_bc(
    base_params=params_base,
    decision_vars={"T_wall": (700.0, 1200.0)},
    objective_fn=lambda col: -(1.0 - col._rho_s_results[-1, 0, 0] / col._rho_s_results[0, 0, 0]),
    solver_config=solver_cfg,
    method="L-BFGS-B",
)
```

### Bloque 4D — Análisis de sensibilidad

Cuantificar el impacto de ±10 % en los parámetros cinéticos (Ea de pirólisis)
sobre la conversión final de biomasa.

```python
from src.control.optimization import sensitivity_analysis

sens = sensitivity_analysis(
    base_params=params_base,
    param_path="fuel_config.kinetics.pyrolysis.Ea",
    delta_pct=0.10,
    objective_fn=lambda col: 1.0 - col._rho_s_results[-1, 0, 0] / col._rho_s_results[0, 0, 0],
    solver_config=solver_cfg,
)
```

---

## Dependencias pendientes

| Módulo | Estado | Descripción |
|--------|--------|-------------|
| `src.control.optimization` | ❌ No implementado | `parametric_sweep`, `optimize_bc`, `sensitivity_analysis` |
| PID runner-level | ❌ Pendiente | Patrón de acumulador integral en `runner_gasifier.py` |
| `src.control.signals.sine` | ✅ Implementado | Disponible — añadir test en bloque 4A |

Para el estado actual del framework de señales, ver `test_gasifier_03_0D_signals.ipynb`.